In [1]:
import sys
import os
sys.path.append("..")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier
from pydvl.influence.torch import CgInfluence
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from arfpy import arf

from bonXAI.core.preprocessor import Preprocessor
from bonXAI.core.explainer import Explainer
from bonXAI.core.evaluation import Evaluator
from bonXAI.core.utils import set_global_seed, possible_g_values, possible_num_bins_values

from openxai.model import LoadModel, ReturnLoaders

SEED = 42
set_global_seed(SEED)

In [3]:
# generate data
# X, y = make_classification(n_samples=20, n_features=7, n_classes=2, random_state=SEED)
# df_data = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
# df_data["label"] = y

# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# or load data
_, loader_test = ReturnLoaders(data_name='compas', download=False, batch_size=128)
X = loader_test.dataset.data
y = loader_test.dataset.targets.to_numpy()

In [9]:
X = loader_test.dataset.data[:20]
y = loader_test.dataset.targets.to_numpy()[:20]

In [10]:
# train model
# model = MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=SEED)
# model.fit(X, y)

# or load model
model = LoadModel(data_name='compas', ml_model='ann', pretrained=True)
model.eval()

ArtificialNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=100, bias=True)
    (3): ReLU()
    (4): Linear(in_features=100, out_features=2, bias=True)
  )
)

In [21]:
# calculate Ground Truth
# gt_explainer = Explainer(model=model, name="shap", variant="kernel", seed=SEED)
# exp_gt, t_gt = gt_explainer.explain(X, y)
# exp_gt_mean = exp_gt.mean(axis=0)

# or load GT
shap_values_gt = np.load("../metadata/compas/gt_shap_sage_results_0_2.npy", allow_pickle=True)
shap_values_gt = shap_values_gt.item()
shap_kernel_list = shap_values_gt['shap_kernel']
shap_kernel_array = np.stack(shap_kernel_list)
exp_gt_mean = shap_kernel_array.mean(axis=0).mean(axis=0)

In [7]:
# save GT explanation
# data_name = "test_data" 
# num_repeats = 1
# save_dir = f"metadata/{data_name}"
# os.makedirs(save_dir, exist_ok=True)
# npz_path = os.path.join(save_dir, f"ground_truth_shap_kernel_{num_repeats}.npz")
# np.savez_compressed(npz_path, explanation_values=exp_gt, explanation_mean=exp_gt_mean, time=t_gt)
# print(f"Saved explanation to {npz_path}")
# csv_path = npz_path.replace(".npz", ".csv")
# df = pd.DataFrame(exp_gt_mean.reshape(1, -1))
# df.to_csv(csv_path, index=False)
# print(f"Saved mean explanation to {csv_path}")

## PyDVL

In [8]:
# compute influence (to get all points' influence set target_size to X size)
pre = Preprocessor(
    method="influence",
    model=model,               # must be a trained PyTorch model
    target_size=3,           
    seed=42                   
)

X_red, y_red, idx, t_comp, influence_matrix = pre.run(X, y)
influence_avg = influence_matrix.mean(axis=0)

In [25]:
results = []

methods = ["iid", "influence", "arfpy", "compress", "compress_with_predictions", "compress_stratified"]
kernels = ["gaussian", "sobolev", "inverse_multiquadric"]
n_samples = X.shape[0]
bins_list = possible_num_bins_values(n_samples)


methods = ["arfpy"]
kernels = ["gaussian"]

evaluator = Evaluator(ground_truth_explanation=exp_gt_mean, reference_points=X)

for method in methods:
    if method == "iid":
        pre = Preprocessor(method=method, model=model, kernel="gaussian", seed=SEED)
        X_red, y_red, idx, t_comp = pre.run(X, y)
        print(f"{method} compression complete")
        for explainer_name, variant in [("shap", "kernel"), ("sage", "permutation")]:
            explainer = Explainer(model=model, name=explainer_name, variant=variant, seed=SEED)
            values, t_exp = explainer.explain(X_red, y_red)
            exp_mean = values
            if exp_mean.ndim > 1:
                exp_mean = values.mean(axis=0)
            row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_red))
            row.update(evaluator.evaluate_compression(X_red))
            row.update({
                "method": method,
                "g": None,
                "num_bins": None,
                "compression_time": t_comp,
                "kernel": None,
                "explainer": explainer_name,
                "variant": variant,
            })
            results.append(row)
            print(f"{method} {explainer_name} {variant} explanation complete")
    elif method == "influence":
        pre = pre = Preprocessor(method="influence", model=model, target_size=5, seed=SEED) # model must be a trained PyTorch model
        X_red, y_red, idx, t_comp, influence_matrix = pre.run(X, y)
        print(f"{method} compression complete")
        for explainer_name, variant in [("shap", "kernel"), ("sage", "permutation")]:
            explainer = Explainer(model=model, name=explainer_name, variant=variant, seed=SEED)
            values, t_exp = explainer.explain(X_red, y_red)
            exp_mean = values
            if exp_mean.ndim > 1:
                exp_mean = values.mean(axis=0)
            row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_red))
            row.update(evaluator.evaluate_compression(X_red))
            row.update({
                "method": method,
                "g": None,
                "num_bins": None,
                "compression_time": t_comp,
                "kernel": None,
                "explainer": explainer_name,
                "variant": variant,
            })
            results.append(row)
            print(f"{method} {explainer_name} {variant} explanation complete")
    elif method == "arfpy":
        pre = Preprocessor(method="arfpy", target_size=5, seed=SEED)
        X_red, y_red, idx, t_comp = pre.run(X, y)
        print(f"{method} compression complete")
        for explainer_name, variant in [("shap", "kernel"), ("sage", "permutation")]:
            explainer = Explainer(model=model, name=explainer_name, variant=variant, seed=SEED)
            values, t_exp = explainer.explain(X_red, y_red)
            exp_mean = values
            if exp_mean.ndim > 1:
                exp_mean = values.mean(axis=0)
            row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_red))
            row.update(evaluator.evaluate_compression(X_red))
            row.update({
                "method": method,
                "g": None,
                "num_bins": None,
                "compression_time": t_comp,
                "kernel": None,
                "explainer": explainer_name,
                "variant": variant,
            })
            results.append(row)
            print(f"{method} {explainer_name} {variant} explanation complete")
    else:
        for kernel in kernels:
            for num_bins in bins_list:
                g_list = possible_g_values(n_samples, num_bins)
                for g in g_list:
                    pre = Preprocessor(method=method, model=model, g=g, num_bins=num_bins, kernel=kernel, seed=SEED)
                    X_red, y_red, idx, t_comp = pre.run(X, y)
                    print(f"{method} compression with g={g}, num_bins={num_bins}, kernel={kernel} complete")

                    for explainer_name, variant in [("shap", "kernel"), ("sage", "permutation")]:
                        explainer = Explainer(model=model, name=explainer_name, variant=variant, seed=SEED)
                        values, t_exp = explainer.explain(X_red, y_red)
                        exp_mean = values
                        if exp_mean.ndim > 1:
                            exp_mean = values.mean(axis=0)
                        row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_red))
                        row.update(evaluator.evaluate_compression(X_red))
                        row.update({
                            "method": method,
                            "g": g,
                            "num_bins": num_bins,
                            "compression_time": t_comp,
                            "kernel": kernel,
                            "explainer": explainer_name,
                            "variant": variant,
                        })
                        results.append(row)
                        print(f"{method} {explainer_name} {variant} with g={g}, num_bins={num_bins}, kernel={kernel} explanation complete")
                        

Initial accuracy is 0.175
arfpy compression complete
arfpy shap kernel explanation complete
PermutationEstimator will use 16 jobs
arfpy sage permutation explanation complete


In [26]:
df_results = pd.DataFrame(results)
df_results.sort_values(by="mae")
df_results

,mae,top_k,time,size,mmd,method,g,num_bins,compression_time,kernel,explainer,variant
0,4.084973e-10,0.8,0.040283,5,0.041619,arfpy,None,None,0.677411,None,shap,kernel
1,6.883643e-02,0.6,1.873822,5,0.041619,arfpy,None,None,0.677411,None,sage,permutation


In [ ]:
data_name = "test_data" 
num_repeats = 1
save_dir = f"metadata/{data_name}"
os.makedirs(save_dir, exist_ok=True)

csv_path = os.path.join(save_dir, f"experiment_results_{num_repeats}.csv")
df_results.to_csv(csv_path, index=False)
print(f"Saved DataFrame to CSV: {csv_path}")